# MatchmakingAgent Test Notebook

This notebook tests the MatchmakingAgent which analyzes grant fit AND checks policy/compliance.

**Model**: Claude Haiku (anthropic.claude-haiku-4-5-20251001-v1:0)

**Purpose**: Dual responsibility - match scoring AND compliance checking

In [ ]:
import sys
import os
import asyncio
import json

sys.path.append(os.path.dirname(os.path.abspath('')))

from agents.matchmaking import run_matchmaking_agent

## Test Case 1: High Match Score Scenario

In [ ]:
# Test data
user_profile = {
    "role": "Assistant Professor",
    "year": "2023",
    "program": "Computer Science"
}

sourced_data = {
    "experience": ["Assistant Professor in Computer Science", "5 years industry experience"],
    "publications": ["Machine Learning in Healthcare", "AI Ethics Framework"],
    "expertise": ["Machine Learning", "Artificial Intelligence", "Data Science"],
    "credentials": ["PhD Computer Science", "NSF CAREER Award"]
}

match_criteria = "Research in AI/ML applications for healthcare, interdisciplinary collaboration"
eligibility = "Assistant or Associate Professor, STEM field, less than 7 years since PhD"

print("Test Case 1: High Match Scenario")
print("User Profile:", json.dumps(user_profile, indent=2))
print("\nMatch Criteria:", match_criteria)
print("\n" + "="*50 + "\n")

In [ ]:
async def test_matchmaking_agent(profile, sourced, criteria, elig):
    results = []
    async for message in run_matchmaking_agent(profile, sourced, criteria, elig):
        results.append(message)
        print(f"[{message['agent']}] {message['step']}")
        if message['done']:
            print("\nFinal Output:")
            output = message['output']
            print(f"Match Score: {output['matchScore']}%")
            print(f"\nJustification: {output['matchJustification']}")
            print(f"\nCompliance Checklist ({len(output['complianceChecklist'])} items):")
            for item in output['complianceChecklist']:
                print(f"  [{item['status'].upper()}] {item['task']} ({item['category']})")
    return results

results_1 = await test_matchmaking_agent(user_profile, sourced_data, match_criteria, eligibility)

## Test Case 2: Moderate Match Score Scenario

In [ ]:
user_profile_2 = {
    "role": "Associate Professor",
    "year": "2018",
    "program": "Biology"
}

sourced_data_2 = {
    "experience": ["Associate Professor in Biology"],
    "publications": ["Genomics Research"],
    "expertise": ["Molecular Biology", "Genetics"],
    "credentials": ["PhD Biology"]
}

match_criteria_2 = "Computational biology and bioinformatics research"
eligibility_2 = "Faculty in life sciences, computational experience preferred"

print("Test Case 2: Moderate Match Scenario")
print("User Profile:", json.dumps(user_profile_2, indent=2))
print("\nMatch Criteria:", match_criteria_2)
print("\n" + "="*50 + "\n")

results_2 = await test_matchmaking_agent(user_profile_2, sourced_data_2, match_criteria_2, eligibility_2)

## Test Case 3: Edge Case - Minimal Profile

In [ ]:
user_profile_3 = {
    "role": "Postdoc",
    "year": "2024",
    "program": "Physics"
}

sourced_data_3 = {
    "experience": [],
    "publications": [],
    "expertise": ["Physics"],
    "credentials": []
}

match_criteria_3 = "Early career researchers in STEM"
eligibility_3 = "Postdoctoral researchers or junior faculty"

print("Test Case 3: Minimal Profile")
print("User Profile:", json.dumps(user_profile_3, indent=2))
print("\nMatch Criteria:", match_criteria_3)
print("\n" + "="*50 + "\n")

results_3 = await test_matchmaking_agent(user_profile_3, sourced_data_3, match_criteria_3, eligibility_3)

## Analyze Compliance Patterns

In [ ]:
# Analyze compliance checklist patterns
print("Compliance Analysis Across Test Cases:\n")

for i, results in enumerate([results_1, results_2, results_3], 1):
    final_output = [r for r in results if r['done']][0]['output']
    checklist = final_output['complianceChecklist']
    
    status_counts = {}
    category_counts = {}
    
    for item in checklist:
        status = item['status']
        category = item['category']
        status_counts[status] = status_counts.get(status, 0) + 1
        category_counts[category] = category_counts.get(category, 0) + 1
    
    print(f"Test Case {i}:")
    print(f"  Match Score: {final_output['matchScore']}%")
    print(f"  Status Distribution: {status_counts}")
    print(f"  Category Distribution: {category_counts}")
    print()